# Outlier Investigation

Notebook 06 revealed a handful of files with unusual dimensions — either far more
pings than the rest (long recordings) or far fewer range samples (short configured
depth). This notebook loads each outlier's echogram at both frequencies, extracts
GPS coordinates from the raw files, and compares their statistics to the rest of
the dataset.

In [ ]:
import sys, gc
sys.path.insert(0, '../../../..')

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import contextily as cx

from raw_sensor_integration.raw_parser import read_raw_file
from raw_sensor_integration.gps_parser import parse_nme0_datagrams, parse_gps_sentences, haversine_m
from raw_sensor_integration.preprocessing import load_sv_from_zarr
from raw_sensor_integration.visualization import plot_echogram, ECHOGRAM_CMAP

DATASETS = Path('/Users/axelhagen/Master/Datasets')
ZARR_DIR = DATASETS / 'echopype_zarr'
RAW_DIR  = DATASETS / 'raw_files'

zarr_stores = sorted(ZARR_DIR.glob('*.zarr'))
print(f'{len(zarr_stores)} zarr stores')

## Collect dimensions from all zarr stores

In [ ]:
records = []
for zp in zarr_stores:
    try:
        ds = xr.open_zarr(str(zp), consolidated=False)
        n_pings = ds.sizes['ping_time']
        n_range = ds.sizes['range_sample']
        ping_times = ds['ping_time'].values
        t_start = pd.Timestamp(ping_times[0])
        t_end   = pd.Timestamp(ping_times[-1])
        duration_min = (t_end - t_start).total_seconds() / 60

        max_depth = np.nan
        if 'echo_range' in ds:
            max_depth = float(ds['echo_range'].isel(channel=0, ping_time=0).values[-1])

        records.append({
            'file': zp.stem, 'n_pings': n_pings, 'n_range_samples': n_range,
            'duration_min': round(duration_min, 2), 'max_depth_m': round(max_depth, 1),
            'start_time': t_start,
        })
        ds.close()
    except Exception as e:
        print(f'  Error: {zp.name}: {e}')

df = pd.DataFrame(records).sort_values('start_time').reset_index(drop=True)
print(f'{len(df)} files loaded')

## Identify outliers

Files are flagged as outliers if their ping count or range sample count falls
outside 1.5× the interquartile range — the standard boxplot criterion.

In [ ]:
def iqr_outliers(series, factor=1.5):
    """Return boolean mask of IQR outliers."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - factor * iqr) | (series > q3 + factor * iqr)

ping_outlier  = iqr_outliers(df['n_pings'])
range_outlier = iqr_outliers(df['n_range_samples'])
is_outlier = ping_outlier | range_outlier

df['outlier_reason'] = ''
df.loc[ping_outlier, 'outlier_reason']  += 'pings '
df.loc[range_outlier, 'outlier_reason'] += 'range_samples'
df['outlier_reason'] = df['outlier_reason'].str.strip()

outliers = df[is_outlier].copy()
print(f'{len(outliers)} outlier files detected out of {len(df)}\n')
outliers[['file', 'n_pings', 'n_range_samples', 'duration_min', 'max_depth_m', 'outlier_reason']]

## Statistics comparison

How the outlier files compare to the rest of the dataset.

In [ ]:
normal = df[~is_outlier]
cols = ['n_pings', 'n_range_samples', 'duration_min', 'max_depth_m']

comparison = pd.DataFrame({
    'Dataset median': normal[cols].median(),
    'Dataset mean':   normal[cols].mean().round(1),
    'Dataset std':    normal[cols].std().round(1),
})
for _, row in outliers.iterrows():
    comparison[row['file']] = row[cols]

comparison

## Echograms — 38 kHz and 200 kHz

Each outlier file plotted at both frequencies. The colour scale is clipped to
the standard Sv range used elsewhere in the project.

In [ ]:
for _, row in outliers.iterrows():
    zarr_path = ZARR_DIR / f"{row['file']}.zarr"

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    for ax, freq, label in zip(axes, [38000, 200000], ['38 kHz', '200 kHz']):
        try:
            sv_db, range_arr, _, _, _, freq_used = load_sv_from_zarr(zarr_path, freq)
            plot_echogram(sv_db, title=f"{row['file']} — {label}",
                          vmin=-80, vmax=-34, ax=ax, range_array=range_arr)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading {label}:\n{e}',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f"{row['file']} — {label}")

    fig.suptitle(
        f"{row['file']}   |   {row['n_pings']} pings × {row['n_range_samples']} samples   |   "
        f"{row['duration_min']:.1f} min   |   outlier: {row['outlier_reason']}",
        fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
    gc.collect()

## GPS locations

For each outlier, the corresponding `.raw` file is located and its GPS track
extracted. The map shows the vessel track during each outlier recording,
with start (▲) and end (■) markers.

In [ ]:
# Find raw file for a given zarr stem — search all subfolders
def find_raw_file(stem):
    for raw_path in RAW_DIR.rglob(f'{stem}.raw'):
        return raw_path
    return None

gps_tracks = {}

for _, row in outliers.iterrows():
    stem = row['file']
    raw_path = find_raw_file(stem)
    if raw_path is None:
        print(f'{stem}: raw file not found')
        continue

    print(f'{stem}: reading {raw_path.name}...', end=' ', flush=True)
    data = read_raw_file(raw_path)
    records = parse_nme0_datagrams(data)
    del data
    gc.collect()

    gps_df = parse_gps_sentences(records)
    del records
    gc.collect()

    if len(gps_df) == 0:
        print('no GPS fixes')
        continue

    gps_df = gps_df.sort_values('timestamp').reset_index(drop=True)
    gps_tracks[stem] = gps_df

    # Distance
    total_m = sum(
        haversine_m(gps_df['lat'].iloc[i], gps_df['lon'].iloc[i],
                    gps_df['lat'].iloc[i+1], gps_df['lon'].iloc[i+1])
        for i in range(len(gps_df) - 1)
    )

    print(f'{len(gps_df)} fixes, {total_m/1000:.2f} km travelled')
    print(f'  start: {gps_df["lat"].iloc[0]:.5f}°N, {gps_df["lon"].iloc[0]:.5f}°E')
    print(f'  end:   {gps_df["lat"].iloc[-1]:.5f}°N, {gps_df["lon"].iloc[-1]:.5f}°E')

## Per-outlier GPS maps

In [ ]:
colors = plt.cm.tab10.colors

for i, (stem, track) in enumerate(gps_tracks.items()):
    fig, ax = plt.subplots(figsize=(10, 8))
    c = colors[i % len(colors)]

    ax.plot(track['lon'], track['lat'], '-', color=c, linewidth=1.5, alpha=0.85, zorder=3)
    ax.plot(track['lon'].iloc[0],  track['lat'].iloc[0],  '^', color=c, markersize=10, zorder=5)
    ax.plot(track['lon'].iloc[-1], track['lat'].iloc[-1], 's', color=c, markersize=10, zorder=5)

    lon_range = track['lon'].max() - track['lon'].min()
    lat_range = track['lat'].max() - track['lat'].min()
    if lon_range < 1e-6 or lat_range < 1e-6:
        pad = 0.01
        ax.set_xlim(track['lon'].mean() - pad, track['lon'].mean() + pad)
        ax.set_ylim(track['lat'].mean() - pad, track['lat'].mean() + pad)
    else:
        cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
                       zoom=13, attribution_size=6)

    row = outliers[outliers['file'] == stem].iloc[0]
    ax.set_title(f'{stem}  —  {row["n_pings"]} pings, {row["duration_min"]:.0f} min  '
                 f'(▲ start, ■ end)', fontsize=11)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Combined GPS map

All outlier tracks on one map for spatial comparison.

In [ ]:
if gps_tracks:
    fig, ax = plt.subplots(figsize=(12, 9))

    for i, (stem, track) in enumerate(gps_tracks.items()):
        c = colors[i % len(colors)]
        ax.plot(track['lon'], track['lat'], '-', color=c, linewidth=2, alpha=0.8,
                label=stem, zorder=3)
        ax.plot(track['lon'].iloc[0],  track['lat'].iloc[0],  '^', color=c,
                markersize=10, zorder=5)
        ax.plot(track['lon'].iloc[-1], track['lat'].iloc[-1], 's', color=c,
                markersize=10, zorder=5)

    all_lats = pd.concat([t['lat'] for t in gps_tracks.values()])
    all_lons = pd.concat([t['lon'] for t in gps_tracks.values()])
    lon_range = all_lons.max() - all_lons.min()
    lat_range = all_lats.max() - all_lats.min()

    if lon_range > 1e-6 and lat_range > 1e-6:
        cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
                       zoom=12, attribution_size=6)

    ax.set_title('All outlier GPS tracks  (▲ start, ■ end)', fontsize=12)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.legend(fontsize=8, loc='upper left', framealpha=0.9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

## Observations

The width outliers are simply longer recordings — the operator did not stop/restart
the file as frequently, so these files accumulated more pings. The echograms
themselves look normal; the acoustic content and bottom structure are consistent
with the rest of the survey. The ping rate is the same (~2 Hz), confirming that
the only difference is recording duration.

The range-sample outlier has a much shallower configured depth, which means the
instrument was set to a shorter maximum range for that segment. This produces
fewer range samples per ping and a smaller echogram height, but the data within
that range is valid.